In [10]:
from bhuvan_flood import layer_name, tile_url, state_config
cfg = state_config('Kerala')
name = layer_name(cfg['code'], '2018-08-16')   # flood:Akl_2018_16_08
print(name)   # Akl_2018_16_08
# print(tile_url(cfg['code'], '2018-08-16'))   # https://tiles.bhuvan-app1.nrsc.gov.in/arcgis/rest/services/Flood/Akl_2018_16_08/ImageServer


flood:Akl_2018_16_08


In [13]:

# Append a known suffix if the bare layer didn't exist that day:
name_06 = layer_name(cfg['code'], '2018-08-16', '_06')
print(tile_url(name_06, 76.0, 10.0, 76.2, 10.2))

https://bhuvan-gp1.nrsc.gov.in/bhuvan/gwc/service/wms?LAYERS=flood%3Akl_2018_16_08_06&TRANSPARENT=TRUE&SERVICE=WMS&VERSION=1.1.1&REQUEST=GetMap&STYLES=&FORMAT=image%2Fpng&SRS=EPSG%3A4326&BBOX=76.0%2C10.0%2C76.2%2C10.2&WIDTH=256&HEIGHT=256


In [12]:
from importlib import reload
import bhuvan_flood, bhuvan_flood.config, bhuvan_flood.wms_client

# Force a fresh load from disk in case Python cached the old module.
reload(bhuvan_flood.config)
reload(bhuvan_flood.wms_client)
reload(bhuvan_flood)

cfg = bhuvan_flood.state_config('Kerala')
print('code from config :', repr(cfg['code']))
print('layer name       :', bhuvan_flood.layer_name(cfg['code'], '2018-08-16', '_06'))

code from config : 'kl'
layer name       : flood:kl_2018_16_08_06


In [14]:
from bhuvan_flood import tile_url, layer_name, state_config, covering_tiles, tile_bbox

cfg = state_config('Kerala')
name = layer_name(cfg['code'], '2018-08-16', '_06')   # flood:kl_2018_16_08_06

# Pick the center tile of Kerala's covering set — a real, grid-aligned bbox
tx_min, ty_min, tx_max, ty_max = covering_tiles(cfg['bbox'])
center_tx = (tx_min + tx_max) // 2
center_ty = (ty_min + ty_max) // 2
w, s, e, n = tile_bbox(center_tx, center_ty)

print(tile_url(name, w, s, e, n))

https://bhuvan-gp1.nrsc.gov.in/bhuvan/gwc/service/wms?LAYERS=flood%3Akl_2018_16_08_06&TRANSPARENT=TRUE&SERVICE=WMS&VERSION=1.1.1&REQUEST=GetMap&STYLES=&FORMAT=image%2Fpng&SRS=EPSG%3A4326&BBOX=75.9375%2C10.546875%2C76.11328125%2C10.72265625&WIDTH=256&HEIGHT=256


In [16]:
#try for a single day


import time
from bhuvan_flood import BhuvanClient, state_config
from bhuvan_flood.stitch import stitch_date
from bhuvan_flood.wms_client import layer_name, tiles_for_bbox

cfg = state_config('Kerala')
client = BhuvanClient()

# Pick a known flood day. Aug 16 2018 had the Kerala flood; the _06 suffix
# is the one your HAR-scraped working URL used for a similar date.
date_iso = '2023-07-06'
name = layer_name(cfg['code'], date_iso, '_06')
tile_count = len(tiles_for_bbox(cfg['bbox']))

print(f"Layer:  {name}")
print(f"Tiles:  {tile_count}")
print()

t0 = time.time()
mask, transform = stitch_date(client, name, cfg['bbox'])
elapsed = time.time() - t0

print(f"Elapsed:       {elapsed:.1f} s ({elapsed/tile_count*1000:.0f} ms/tile)")
print(f"Mask shape:    {mask.shape}")
print(f"Flood pixels:  {int(mask.sum()):,}  ({100*mask.sum()/mask.size:.2f}% of area)")
print()
print(f"Extrapolated for 140 days: ~{elapsed*140/60:.0f} min worst case")

Layer:  flood:kl_2023_06_07_06
Tiles:  504

Elapsed:       115374.8 s (228918 ms/tile)
Mask shape:    (7168, 4608)
Flood pixels:  0  (0.00% of area)

Extrapolated for 140 days: ~269208 min worst case
